In [8]:
# Celda 1 — imports
import sys
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

sys.path.append(str(Path('..') / 'src'))
from config import CH, BRONZE_DB, SILVER_DB

In [13]:
def cargar_a_silver(nombre_tabla, query):
    print(f"Leyendo bronze.raw_{nombre_tabla}...")
    result = CH.query(query)
    rows = result.result_rows
    cols = result.column_names

    df = pd.DataFrame(rows, columns=cols)

    df['_processed_at'] = datetime.now()
    df = df.where(pd.notnull(df), None)

    
    if nombre_tabla == 'rental':
        df['return_date'] = df['return_date'].fillna(datetime(1970, 1, 1))

    data = []
    for row in df.values.tolist():
        clean_row = []
        for val in row:
            if isinstance(val, float) and np.isnan(val):
                clean_row.append(None)
            elif isinstance(val, (np.integer,)):
                clean_row.append(int(val))
            elif isinstance(val, (np.floating,)):
                clean_row.append(float(val))
            else:
                clean_row.append(val)
        data.append(clean_row)

    CH.command(f"TRUNCATE TABLE {SILVER_DB}.{nombre_tabla}")
    CH.insert(f"{SILVER_DB}.{nombre_tabla}",
              data,
              column_names=df.columns.tolist())
    print(f"✓ {len(df):,} filas cargadas en silver.{nombre_tabla}")

In [4]:
tablas_sakila = {
    'actor': f"""
        SELECT
            actor_id,
            trimBoth(ifNull(first_name, '')) AS first_name,
            trimBoth(ifNull(last_name,  '')) AS last_name,
            last_update
        FROM {BRONZE_DB}.raw_actor
    """,

    'address': f"""
        SELECT
            address_id,
            trimBoth(ifNull(address,      '')) AS address,
            trimBoth(ifNull(address2,     '')) AS address2,
            trimBoth(ifNull(district,     '')) AS district,
            ifNull(city_id, 0)                 AS city_id,
            trimBoth(ifNull(postal_code,  '')) AS postal_code,
            trimBoth(ifNull(phone,        '')) AS phone,
            last_update
        FROM {BRONZE_DB}.raw_address
    """,

    'category': f"""
        SELECT
            category_id,
            trimBoth(ifNull(name, '')) AS name,
            last_update
        FROM {BRONZE_DB}.raw_category
    """,

    'city': f"""
        SELECT
            city_id,
            trimBoth(ifNull(city, '')) AS city,
            ifNull(country_id, 0)      AS country_id,
            last_update
        FROM {BRONZE_DB}.raw_city
    """,

    'country': f"""
        SELECT
            country_id,
            trimBoth(ifNull(country, '')) AS country,
            last_update
        FROM {BRONZE_DB}.raw_country
    """,

    'customer': f"""
        SELECT
            customer_id,
            ifNull(store_id, 0)                AS store_id,
            trimBoth(ifNull(first_name, ''))   AS first_name,
            trimBoth(ifNull(last_name,  ''))   AS last_name,
            trimBoth(ifNull(email,      ''))   AS email,
            ifNull(address_id, 0)              AS address_id,
            ifNull(active, 0)                  AS active,
            create_date,
            last_update
        FROM {BRONZE_DB}.raw_customer
    """,

    'film': f"""
        SELECT
            film_id,
            trimBoth(ifNull(title, ''))            AS title,
            trimBoth(ifNull(description, ''))      AS description,
            ifNull(release_year, 0)                AS release_year,
            ifNull(language_id, 0)                 AS language_id,
            ifNull(original_language_id, 0)        AS original_language_id,
            ifNull(rental_duration, 0)             AS rental_duration,
            ifNull(rental_rate, 0.0)               AS rental_rate,
            ifNull(length, 0)                      AS length,
            ifNull(replacement_cost, 0.0)          AS replacement_cost,
            trimBoth(ifNull(rating, ''))           AS rating,
            trimBoth(ifNull(special_features, '')) AS special_features,
            last_update
        FROM {BRONZE_DB}.raw_film
    """,

    'film_actor': f"""
        SELECT
            actor_id,
            film_id,
            last_update
        FROM {BRONZE_DB}.raw_film_actor
    """,

    'film_category': f"""
        SELECT
            film_id,
            category_id,
            last_update
        FROM {BRONZE_DB}.raw_film_category
    """,

    'inventory': f"""
        SELECT
            inventory_id,
            ifNull(film_id, 0)  AS film_id,
            ifNull(store_id, 0) AS store_id,
            last_update
        FROM {BRONZE_DB}.raw_inventory
    """,

    'language': f"""
        SELECT
            language_id,
            trimBoth(ifNull(name, '')) AS name,
            last_update
        FROM {BRONZE_DB}.raw_language
    """,

    'payment': f"""
        SELECT
            payment_id,
            ifNull(customer_id, 0)   AS customer_id,
            ifNull(staff_id, 0)      AS staff_id,
            ifNull(rental_id, 0)     AS rental_id,
            ifNull(amount, 0.0)      AS amount,
            payment_date,
            last_update
        FROM {BRONZE_DB}.raw_payment
    """,

    'rental': f"""
        SELECT
            rental_id,
            rental_date,
            ifNull(inventory_id, 0) AS inventory_id,
            ifNull(customer_id, 0)  AS customer_id,
            return_date,
            ifNull(staff_id, 0)     AS staff_id,
            last_update
        FROM {BRONZE_DB}.raw_rental
    """,

    'staff': f"""
        SELECT
            staff_id,
            trimBoth(ifNull(first_name, '')) AS first_name,
            trimBoth(ifNull(last_name,  '')) AS last_name,
            ifNull(address_id, 0)            AS address_id,
            trimBoth(ifNull(email,      '')) AS email,
            ifNull(store_id, 0)              AS store_id,
            ifNull(active, 0)                AS active,
            trimBoth(ifNull(username,   '')) AS username,
            last_update
        FROM {BRONZE_DB}.raw_staff
    """,

    'store': f"""
        SELECT
            store_id,
            ifNull(manager_staff_id, 0) AS manager_staff_id,
            ifNull(address_id, 0)       AS address_id,
            last_update
        FROM {BRONZE_DB}.raw_store
    """
}

In [14]:
# Celda 4 — ejecutar
for tabla, query in tablas_sakila.items():
    cargar_a_silver(tabla, query)

Leyendo bronze.raw_actor...
✓ 200 filas cargadas en silver.actor
Leyendo bronze.raw_address...
✓ 603 filas cargadas en silver.address
Leyendo bronze.raw_category...
✓ 16 filas cargadas en silver.category
Leyendo bronze.raw_city...
✓ 600 filas cargadas en silver.city
Leyendo bronze.raw_country...
✓ 109 filas cargadas en silver.country
Leyendo bronze.raw_customer...
✓ 599 filas cargadas en silver.customer
Leyendo bronze.raw_film...
✓ 1,000 filas cargadas en silver.film
Leyendo bronze.raw_film_actor...
✓ 5,462 filas cargadas en silver.film_actor
Leyendo bronze.raw_film_category...
✓ 1,000 filas cargadas en silver.film_category
Leyendo bronze.raw_inventory...
✓ 4,581 filas cargadas en silver.inventory
Leyendo bronze.raw_language...
✓ 6 filas cargadas en silver.language
Leyendo bronze.raw_payment...
✓ 14,881 filas cargadas en silver.payment
Leyendo bronze.raw_rental...
✓ 16,044 filas cargadas en silver.rental
Leyendo bronze.raw_staff...
✓ 2 filas cargadas en silver.staff
Leyendo bronze.raw_